# A2A MCP Client

Test client for the orchestrator agent. Run all agent notebooks first:
1. `mcp_server.ipynb` (port 10100)
2. `planner_agent.ipynb` (port 10102)
3. `travel_agents.ipynb` (ports 10103-10105)
4. `orchestrator_agent.ipynb` (port 10101)

In [1]:
import json

import httpx
from a2a.client import ClientConfig, ClientFactory
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest,
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatusUpdateEvent,
)
from google.protobuf import json_format

ORCHESTRATOR_URL = 'http://localhost:10101'

## Fetch Agent Card

In [2]:
import httpx

resp = httpx.get(f'{ORCHESTRATOR_URL}/.well-known/agent-card.json')
resp.raise_for_status()
print(json.dumps(resp.json(), indent=2))

{
  "name": "Orchestrator Agent",
  "description": "Orchestrates the task generation and execution",
  "supportedInterfaces": [
    {
      "url": "http://localhost:10101/",
      "protocolBinding": "JSONRPC"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": true
  },
  "defaultInputModes": [
    "text",
    "text/plain"
  ],
  "defaultOutputModes": [
    "text",
    "text/plain"
  ],
  "skills": [
    {
      "id": "executor",
      "name": "Task Executor",
      "description": "Orchestrates the task generation and execution, takes help from the planner to generate tasks",
      "tags": [
        "execute plan"
      ],
      "examples": [
        "Plan my trip to London, submit an expense report"
      ]
    }
  ],
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3",
  "url": "http://localhost:10101/"
}


## Send Streaming Message

In [3]:
from uuid import uuid4


def _parse_input_required_json(text):
    """If text is a code-fenced or raw JSON with status=input_required, return the question."""
    if not text:
        return None
    stripped = text.strip()
    if stripped.startswith('```'):
        stripped = stripped.strip('`')
        if stripped.lower().startswith('json'):
            stripped = stripped[4:]
    try:
        data = json.loads(stripped)
    except (json.JSONDecodeError, ValueError):
        return None
    if isinstance(data, dict) and data.get('status') == 'input_required':
        return data.get('question') or data.get('content')
    return None


async def send_message(text: str, task_id: str = None, context_id: str = None, turn: int = 0):
    """Send one turn to the orchestrator and drain the stream.

    Returns (task_id, context_id, pending_question, completed).
    """
    print(f'\n===== turn {turn}: sending =====')
    print(f'  task_id    = {task_id}')
    print(f'  context_id = {context_id}')
    print(f'  text       = {text!r}')

    httpx_client = httpx.AsyncClient(timeout=httpx.Timeout(None))
    config = ClientConfig(httpx_client=httpx_client, streaming=True)
    client = await ClientFactory.connect(ORCHESTRATOR_URL, client_config=config)

    message = Message(
        role=Role.ROLE_USER,
        parts=[Part(text=text)],
        message_id=uuid4().hex,
    )
    if task_id:
        message.task_id = task_id
    if context_id:
        message.context_id = context_id

    request = SendMessageRequest(message=message)
    pending_question = None
    completed = False
    try:
        try:
            async for stream_resp, _task in client.send_message(request):
                payload_type = stream_resp.WhichOneof('payload')

                if payload_type == 'task':
                    t = stream_resp.task
                    if t.id:
                        task_id = t.id
                    if t.context_id:
                        context_id = t.context_id
                    print(f'[task] id={t.id} context_id={t.context_id} state={TaskState.Name(t.status.state)}')

                elif payload_type == 'msg':
                    m = stream_resp.msg
                    if m.task_id:
                        task_id = m.task_id
                    if m.context_id:
                        context_id = m.context_id
                    msg_text = m.content[0].text if m.content else ''
                    print(f'[msg] {msg_text}')
                    q = _parse_input_required_json(msg_text)
                    if q:
                        pending_question = q

                elif payload_type == 'status_update':
                    evt = stream_resp.status_update
                    if evt.task_id:
                        task_id = evt.task_id
                    if evt.context_id:
                        context_id = evt.context_id
                    msg_text = evt.status.message.parts[0].text if evt.status.message.parts else ''
                    print(f'[{TaskState.Name(evt.status.state)}] {msg_text}')
                    if evt.status.state == TaskState.TASK_STATE_INPUT_REQUIRED:
                        pending_question = msg_text
                    else:
                        q = _parse_input_required_json(msg_text)
                        if q:
                            pending_question = q
                    if evt.status.state == TaskState.TASK_STATE_COMPLETED:
                        completed = True

                elif payload_type == 'artifact_update':
                    au = stream_resp.artifact_update
                    if au.task_id:
                        task_id = au.task_id
                    if au.context_id:
                        context_id = au.context_id
                    artifact = au.artifact
                    print(f'[artifact] {artifact.name}')
                    for part in artifact.parts:
                        content_type = part.WhichOneof('content')
                        if content_type == 'text':
                            print(part.text)
                        elif content_type == 'data':
                            print(json.dumps(json_format.MessageToDict(part.data), indent=2))
        except Exception as exc:
            if pending_question is None:
                raise
            print(f'[stream closed after question: {type(exc).__name__}]')
    finally:
        await httpx_client.aclose()

    print(f'  -> returned task_id={task_id} context_id={context_id} '
          f'pending_question={bool(pending_question)} completed={completed}')
    return task_id, context_id, pending_question, completed


async def run_conversation(initial_text: str, max_turns: int = 10):
    """Drive a full multi-turn conversation, prompting for follow-ups inline."""
    task_id, context_id = None, None
    text = initial_text
    for turn in range(max_turns):
        task_id, context_id, question, done = await send_message(text, task_id, context_id, turn=turn)
        if done:
            print('\n[conversation complete]')
            return
        if question is None:
            print('\n[stream ended without a question and without completion - stopping]')
            return
        print(f'\n>>> Agent asks: {question}')
        text = input('Your answer: ').strip()
        if not text:
            print('[empty answer - stopping]')
            return
    print(f'\n[hit max_turns={max_turns} without completion - stopping]')

In [4]:
await run_conversation('Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in a hotel suite and flying premium economy')


===== turn 0: sending =====
  task_id    = None
  context_id = None
  text       = 'Plan my trip to London from San Francisco from 2026-05-01 to 05-11 - staying in a hotel suite and flying premium economy'
[task] id=0b28a8e4-e80d-4fe7-a8d7-bbc5d4a4d84a context_id=a6c11225-7f91-433a-b60a-75304ed758fc state=TASK_STATE_SUBMITTED
[TASK_STATE_WORKING] ```json
{
    "status": "input_required",
    "question": "What is your total budget for this trip?"
}
```
[TASK_STATE_WORKING] ```json
{
    "status": "input_required",
    "question": "What is your total budget for this trip?"
}
```
[TASK_STATE_INPUT_REQUIRED] What is your total budget for this trip?
  -> returned task_id=0b28a8e4-e80d-4fe7-a8d7-bbc5d4a4d84a context_id=a6c11225-7f91-433a-b60a-75304ed758fc pending_question=True completed=False

>>> Agent asks: What is your total budget for this trip?


Your answer:  30000



===== turn 1: sending =====
  task_id    = 0b28a8e4-e80d-4fe7-a8d7-bbc5d4a4d84a
  context_id = a6c11225-7f91-433a-b60a-75304ed758fc
  text       = '30000'
[TASK_STATE_WORKING] ```json
{
    "status": "input_required",
    "question": "Is this trip for business or leisure?"
}
```
[TASK_STATE_WORKING] ```json
{
    "status": "input_required",
    "question": "Is this trip for business or leisure?"
}
```
[TASK_STATE_INPUT_REQUIRED] How many travelers will be on this trip?
  -> returned task_id=0b28a8e4-e80d-4fe7-a8d7-bbc5d4a4d84a context_id=a6c11225-7f91-433a-b60a-75304ed758fc pending_question=True completed=False

>>> Agent asks: How many travelers will be on this trip?


Your answer:  2



===== turn 2: sending =====
  task_id    = 0b28a8e4-e80d-4fe7-a8d7-bbc5d4a4d84a
  context_id = a6c11225-7f91-433a-b60a-75304ed758fc
  text       = '2'
[TASK_STATE_WORKING] ```json
{
    "status": "input_required",
    "question": "How many travelers will be on this trip?"
}
```
[TASK_STATE_WORKING] ```json
{
    "status": "input_required",
    "question": "How many travelers will be on this trip?"
}
```
[artifact] PlannerAgent-result
{
  "tasks": [],
  "trip_info": {
    "end_date": "2026-05-11",
    "checkin_date": "2026-05-01",
    "type": "leisure",
    "room_type": "suite",
    "travel_class": "premium economy",
    "origin": "San Francisco",
    "no_of_travellers": null,
    "type_of_car": null,
    "is_car_rental_required": null,
    "destination": "London",
    "checkout_date": "2026-05-11",
    "accommodation_type": "hotel",
    "car_rental_end_date": "2026-05-11",
    "total_budget": "30000",
    "start_date": "2026-05-01",
    "car_rental_start_date": "2026-05-01"
  },
  "or